In [ ]:
import torch
import torch.nn as nn
from sklearn.metrics import cohen_kappa_score, confusion_matrix, classification_report
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

def calculate_medical_metrics(all_labels, all_preds):
    # Convert to numpy for sklearn
    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    
    kappa = cohen_kappa_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)
    
    # Extract TN, FP, FN, TP
    # Assuming 0 = Normal, 1 = Fracture
    tn, fp, fn, tp = cm.ravel()
    
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    return {
        "kappa": kappa,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "cm": cm
    }

In [ ]:
class XrayResearchClassifier(nn.Module):
    def __init__(self, checkpoint_path, num_classes=2):
        super().__init__()
        # 1. Load the exact same base used in pre-training
        self.encoder = timm.create_model('vit_base_patch16_224.mae', pretrained=False)
        
        # 2. Load the "Intelligence" you built
        print(f"Injecting Pre-trained Anatomy Knowledge from {checkpoint_path}...")
        state_dict = torch.load(checkpoint_path, map_location="cpu")
        # strict=False is KEY because we don't need the MAE decoder anymore
        self.encoder.load_state_dict(state_dict, strict=False)
        
        # 3. The Classification Head
        # ViT Base usually has 768 features
        self.head = nn.Sequential(
            nn.LayerNorm(self.encoder.num_features),
            nn.Linear(self.encoder.num_features, num_classes)
        )

    def forward(self, x):
        # Extract features (the "Eyes" look at the image)
        features = self.encoder.forward_features(x) 
        # features shape: [Batch, Tokens, 768] -> Mean over tokens
        x = features.mean(dim=1) 
        return self.head(x)

In [ ]:
# Assuming 0: Normal, 1: Fracture
# If you have 3x more Normal than Fracture, use weights=[1.0, 3.0]
class_weights = torch.tensor([1.0, 2.5]).to(DEVICE) 
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
    return total_loss / len(loader), all_labels, all_preds

In [ ]:
def validate_and_plot(model, loader, criterion, epoch):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    metrics = calculate_medical_metrics(all_labels, all_preds)
    
    # Plotting Confusion Matrix
    plt.figure(figsize=(6,5))
    sns.heatmap(metrics['cm'], annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Normal', 'Fracture'], yticklabels=['Normal', 'Fracture'])
    plt.title(f'Confusion Matrix - Epoch {epoch}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()
    
    print(f"Kappa: {metrics['kappa']:.4f} | Sensitivity: {metrics['sensitivity']:.4f} | Specificity: {metrics['specificity']:.4f}")
    return metrics

In [ ]:
import torch.nn.functional as F

class XraySegmentationModel(nn.Module):
    def __init__(self, checkpoint_path):
        super().__init__()
        # 1. Use the Pre-trained Encoder
        self.encoder = timm.create_model('vit_base_patch16_224.mae', pretrained=False)
        
        if checkpoint_path:
            state_dict = torch.load(checkpoint_path, map_location="cpu")
            self.encoder.load_state_dict(state_dict, strict=False)
        
        # 2. Decoder Layers: Upsampling the tokens back to 224x224
        # ViT Base has 768 features per patch. We project this to a smaller channel size.
        self.decoder_conv = nn.Sequential(
            nn.Conv2d(768, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True), # 14 -> 28
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=4, mode='bilinear', align_corners=True), # 28 -> 112
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True), # 112 -> 224
            nn.Conv2d(64, 1, kernel_size=1) # Final mask output (1 channel for binary mask)
        )

    def forward(self, x):
        # Extract features [B, 197, 768] (196 patches + 1 class token)
        features = self.encoder.forward_features(x)
        
        # Remove class token and reshape [B, 196, 768] -> [B, 768, 14, 14]
        features = features[:, 1:, :].transpose(1, 2).reshape(-1, 768, 14, 14)
        
        # Pass through decoder to get the mask [B, 1, 224, 224]
        mask = self.decoder_conv(features)
        return torch.sigmoid(mask)

In [ ]:
def dice_coefficient(preds, targets, smooth=1e-6):
    preds = (preds > 0.5).float()
    intersection = (preds * targets).sum()
    return (2. * intersection + smooth) / (preds.sum() + targets.sum() + smooth)

def iou_score(preds, targets, smooth=1e-6):
    preds = (preds > 0.5).float()
    intersection = (preds * targets).sum()
    union = preds.sum() + targets.sum() - intersection
    return (intersection + smooth) / (union + smooth)

In [ ]:
class DiceBCELoss(nn.Module):
    def __init__(self):
        super(DiceBCELoss, self).__init__()

    def forward(self, inputs, targets):
        inputs = inputs.view(-1)
        targets = targets.view(-1)
        
        bce = F.binary_cross_entropy(inputs, targets, reduction='mean')
        
        # Dice Calculation
        intersection = (inputs * targets).sum()                            
        dice_loss = 1 - (2.*intersection + 1e-6)/(inputs.sum() + targets.sum() + 1e-6)  
        
        return bce + dice_loss

# Initialize for Segmentation
seg_model = XraySegmentationModel("mae_bone_features_final.pth").to(DEVICE)
seg_criterion = DiceBCELoss()
seg_optimizer = torch.optim.AdamW(seg_model.parameters(), lr=1e-4)

In [ ]:
import torch.nn.functional as F

def get_gradcam_heatmap(model, input_image, target_class=1):
    model.eval()
    # We hook into the last block of the ViT Encoder
    # For ViT-Base, this is usually model.encoder.blocks[-1]
    last_block = model.encoder.blocks[-1].norm1
    
    # Storage for gradients and activations
    gradients = []
    activations = []

    def save_gradient(grad): gradients.append(grad)
    def save_activation(module, input, output): activations.append(output)

    # Register hooks
    handle_g = last_block.register_full_backward_hook(lambda m, i, o: save_gradient(o[0]))
    handle_a = last_block.register_forward_hook(save_activation)

    # Forward pass
    output = model(input_image.unsqueeze(0).to(DEVICE))
    loss = output[0, target_class]
    
    # Backward pass
    model.zero_grad()
    loss.backward()

    # Process Heatmap
    # GAP of gradients * activations
    weights = torch.mean(gradients[0], dim=1, keepdim=True)
    heatmap = torch.sum(weights * activations[0], dim=-1).squeeze(0)
    heatmap = F.relu(heatmap) # Keep only positive contributions
    
    # Cleanup
    handle_g.remove()
    handle_a.remove()
    
    return heatmap.detach().cpu().numpy()

In [ ]:
def print_research_summary(metrics, dice_scores, iou_scores):
    print("="*30)
    print("BONE FRACTURE RESEARCH SUMMARY")
    print("="*30)
    print(f"Classification Kappa:      {metrics['kappa']:.4f}")
    print(f"Clinical Sensitivity:      {metrics['sensitivity']:.4f}")
    print(f"Clinical Specificity:      {metrics['specificity']:.4f}")
    print("-"*30)
    print(f"Mean Dice Coefficient:     {np.mean(dice_scores):.4f}")
    print(f"Mean IoU (Jaccard Index):  {np.mean(iou_scores):.4f}")
    print("="*30)

In [ ]:
import cv2

def research_inference(img_path, classifier, segmentation_model, transform):
    # 1. Preprocess the Image
    raw_img = Image.open(img_path).convert("RGB")
    input_tensor = transform(raw_img).to(DEVICE)
    
    # 2. Classification Prediction
    classifier.eval()
    with torch.no_grad():
        logits = classifier(input_tensor.unsqueeze(0))
        probs = F.softmax(logits, dim=1)
        conf, pred = torch.max(probs, 1)
        label = "FRACTURE" if pred.item() == 1 else "NORMAL"

    # 3. Segmentation Mask Generation
    segmentation_model.eval()
    with torch.no_grad():
        mask = segmentation_model(input_tensor.unsqueeze(0))
        mask = (mask > 0.5).float().cpu().squeeze().numpy()

    # 4. Explainability (Grad-CAM)
    heatmap = get_gradcam_heatmap(classifier, input_tensor, target_class=pred.item())
    
    # 5. VISUALIZATION
    plt.figure(figsize=(20, 5))
    
    # Original
    plt.subplot(1, 4, 1)
    plt.imshow(raw_img)
    plt.title(f"Original X-ray\nPred: {label} ({conf.item()*100:.1f}%)")
    plt.axis('off')

    # Segmentation
    plt.subplot(1, 4, 2)
    plt.imshow(raw_img)
    plt.imshow(mask, alpha=0.5, cmap='Reds') # Overlay red mask
    plt.title("AI Segmentation Mask")
    plt.axis('off')

    # Grad-CAM Heatmap
    plt.subplot(1, 4, 3)
    # Resize heatmap to match image
    heatmap_resized = cv2.resize(heatmap, (224, 224))
    plt.imshow(raw_img.resize((224, 224)))
    plt.imshow(heatmap_resized, alpha=0.5, cmap='jet')
    plt.title("AI Attention (Grad-CAM)")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

# Example Usage:
# research_inference("test_xray.jpg", classifier_model, seg_model, val_transform)